# Vulnerability Score Construction

This notebook starts from the final integrated county-year feature dataset created in Notebook 04.

The goal is to construct climate-housing vulnerability sub-scores, combine them into one final vulnerability score, and create Low / Medium / High vulnerability classes for later machine learning.

## 1. Notebook Setup

This section imports the required libraries, sets the project paths, and loads the final NFIP-enhanced county-year feature dataset.

In [1]:
# ==================================================
# Imports
# ==================================================

from pathlib import Path

import pandas as pd
import numpy as np

# ==================================================
# Project Paths
# ==================================================

PROJECT_ROOT = Path("..")

INTERIM_DATA = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "county_year_features"
)

RESULTS_TABLES = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

RESULTS_FIGURES = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

RESULTS_LOGS = (
    PROJECT_ROOT
    / "results"
    / "logs"
)

# ==================================================
# Input Dataset
# ==================================================

INPUT_DATA_PATH = (
    INTERIM_DATA
    / "county_year_housing_spatial_disaster_acs_nfip_features.csv"
)

# ==================================================
# Project Settings
# ==================================================

PROJECT_START_YEAR = 2011
PROJECT_END_YEAR = 2025

## 2. Load Final Integrated Dataset

This section loads the final integrated county-year dataset created in Notebook 04.

This dataset already contains the housing, spatial, disaster-history, ACS socioeconomic, affordability, and NFIP insurance-loss features. It will be used as the starting point for vulnerability score construction.

In [2]:
# ==================================================
# 2. Load Final Integrated Dataset
# ==================================================

county_year_features = pd.read_csv(INPUT_DATA_PATH)

print("Dataset loaded successfully.")
print(f"Shape: {county_year_features.shape}")

display(county_year_features.head())

Dataset loaded successfully.
Shape: (1000, 75)


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,nfip_total_icc_payment,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_total_building_coverage,nfip_total_contents_coverage,nfip_cumulative_claim_count,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_count,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,0.0,0.00,0.000000,0.0,0.0,0.0,0.00,0.0,0.00,0
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,0.0,200195.98,20019.598000,1767900.0,448500.0,10.0,200195.98,10.0,200195.98,1
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,0.0,5630.43,1876.810000,376000.0,132100.0,13.0,205826.41,13.0,205826.41,1
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,0.0,12849.49,6424.745000,500000.0,200000.0,15.0,218675.90,15.0,218675.90,1
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2015,153750.122064,...,0.0,133664.06,22277.343333,1172900.0,334500.0,21.0,352339.96,11.0,152143.98,1


## 3. Basic Dataset Validation

Before creating vulnerability scores, the dataset structure is checked again to make sure the correct file has been loaded.

The key checks are:

- number of rows and columns
- number of Florida counties
- project year range
- duplicate county-year rows
- missing values

In [3]:
# ==================================================
# 3. Basic Dataset Validation
# ==================================================

print("Dataset shape:")
print(county_year_features.shape)

print("\nUnique counties:")
print(county_year_features["STCOFIPS"].nunique())

print("\nYear range:")
print(
    county_year_features["Year"].min(),
    "to",
    county_year_features["Year"].max()
)

duplicate_county_year_rows = county_year_features.duplicated(
    subset=["STCOFIPS", "Year"]
).sum()

print("\nDuplicate county-year rows:")
print(duplicate_county_year_rows)

missing_values = (
    county_year_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_values = missing_values[missing_values > 0]

print("\nColumns with missing values:")
display(missing_values)

Dataset shape:
(1000, 75)

Unique counties:
67

Year range:
2011 to 2025

Duplicate county-year rows:
0

Columns with missing values:


price_growth_acceleration     70
annual_price_growth_dollar     3
prev_year_housing_price        3
annual_price_growth_pct        3
high_growth_flag               3
dtype: int64

## 4. Inspect Score Variables

Check that the variables needed for each vulnerability sub-score are available in the final dataset.

In [5]:
# ==================================================
# 4. Inspect Score Variables
# ==================================================

score_variables = {
    "housing_pressure": [
        "annual_price_growth_pct",
        "appreciation_from_baseline_pct",
        "annual_price_volatility",
        "price_growth_acceleration"
    ],

    "affordability_stress": [
        "price_to_income_ratio",
        "median_gross_rent",
        "median_household_income"
    ],

    "climate_exposure": [
        "CFLD_RISKS",
        "HRCN_RISKS"
    ],

    "disaster_history": [
        "climate_disaster_count",
        "cumulative_climate_disaster_count",
        "recent_3yr_climate_disaster_count",
        "hurricane_disaster_count"
    ],

    "socioeconomic_vulnerability": [
        "SOVI_SCORE",
        "poverty_rate",
        "unemployment_rate",
        "renter_occupied_share"
    ],

    "resilience_adjustment": [
        "RESL_SCORE"
    ],

    "insurance_loss_stress": [
        "nfip_claim_count",
        "nfip_total_claim_payment",
        "nfip_cumulative_claim_payment",
        "nfip_recent_3yr_claim_payment",
        "nfip_avg_claim_payment"
    ],

    "spatial_spillover": [
        "neighbor_avg_price_growth_pct",
        "neighbor_avg_price_volatility",
        "neighbor_high_growth_share",
        "neighbor_high_volatility_share"
    ]
}

for group, variables in score_variables.items():
    missing = [var for var in variables if var not in county_year_features.columns]
    
    print(f"{group}:")
    print(f"  variables: {len(variables)}")
    print(f"  missing: {missing}")

housing_pressure:
  variables: 4
  missing: []
affordability_stress:
  variables: 3
  missing: []
climate_exposure:
  variables: 2
  missing: []
disaster_history:
  variables: 4
  missing: []
socioeconomic_vulnerability:
  variables: 4
  missing: []
resilience_adjustment:
  variables: 1
  missing: []
insurance_loss_stress:
  variables: 5
  missing: []
spatial_spillover:
  variables: 4
  missing: []


## 5. Check Missing Values in Score Variables

Check missing values only for the variables that will be used in vulnerability scoring.

In [6]:
# ==================================================
# 5. Check Missing Values in Score Variables
# ==================================================

all_score_vars = []

for variables in score_variables.values():
    all_score_vars.extend(variables)

all_score_vars = list(dict.fromkeys(all_score_vars))

score_missing_values = (
    county_year_features[all_score_vars]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

score_missing_values = score_missing_values[score_missing_values > 0]

print("Missing values in score variables:")
display(score_missing_values)

Missing values in score variables:


price_growth_acceleration    70
annual_price_growth_pct       3
dtype: int64

## 6. Handle Missing Score Values

Fill missing growth-related score variables before creating standardized scores.

In [7]:
# ==================================================
# 6. Handle Missing Score Values
# ==================================================

vulnerability_df = county_year_features.copy()

score_fill_cols = [
    "annual_price_growth_pct",
    "price_growth_acceleration"
]

print("Missing values before filling:")
print(vulnerability_df[score_fill_cols].isna().sum())

vulnerability_df[score_fill_cols] = vulnerability_df[score_fill_cols].fillna(0)

print("\nMissing values after filling:")
print(vulnerability_df[score_fill_cols].isna().sum())

Missing values before filling:
annual_price_growth_pct       3
price_growth_acceleration    70
dtype: int64

Missing values after filling:
annual_price_growth_pct      0
price_growth_acceleration    0
dtype: int64


## 7. Check Score Variable Distributions

Check the scale and skewness of score variables before standardizing them.

In [8]:
# ==================================================
# 7. Check Score Variable Distributions
# ==================================================

score_summary = (
    vulnerability_df[all_score_vars]
    .describe()
    .T
    .round(3)
)

score_summary["skew"] = (
    vulnerability_df[all_score_vars]
    .skew()
    .round(3)
)

display(score_summary)

,count,mean,std,min,25%,50%,75%,max,skew
annual_price_growth_pct,1000.0,6.271000e+00,7.948000e+00,-14.023,1.316,5.903,9.865000e+00,3.573600e+01,0.510
appreciation_from_baseline_pct,1000.0,5.704500e+01,6.167900e+01,-19.571,3.414,39.777,1.024120e+02,2.670240e+02,0.837
annual_price_volatility,1000.0,5.510032e+03,5.890973e+03,161.593,2161.468,3834.138,6.172371e+03,5.793892e+04,3.388
price_growth_acceleration,1000.0,3.100000e-01,7.387000e+00,-31.271,-2.266,0.055,3.969000e+00,1.918300e+01,-1.090
price_to_income_ratio,1000.0,3.903000e+00,1.261000e+00,1.855,2.970,3.728,4.544000e+00,1.181500e+01,1.670
median_gross_rent,1000.0,9.958170e+02,3.130970e+02,532.000,755.000,937.500,1.143000e+03,2.098700e+03,1.072
median_household_income,1000.0,5.271089e+04,1.419713e+04,29806.000,41522.500,49682.000,6.077950e+04,1.185591e+05,0.947
CFLD_RISKS,1000.0,4.753600e+01,3.772900e+01,0.000,0.000,61.200,7.855000e+01,9.960000e+01,-0.106
HRCN_RISKS,1000.0,9.401500e+01,5.436000e+00,74.468,90.572,94.994,9.866500e+01,9.995800e+01,-1.166
climate_disaster_count,1000.0,1.193000e+00,1.493000e+00,0.000,0.000,1.000,2.000000e+00,7.000000e+00,1.524


## 8. Transform Skewed Score Variables

Apply log transformation to highly skewed positive variables before scaling.

In [9]:
# ==================================================
# 8. Transform Skewed Score Variables
# ==================================================

score_df = vulnerability_df.copy()

log_transform_vars = [
    "annual_price_volatility",
    "price_to_income_ratio",
    "median_gross_rent",
    "neighbor_avg_price_volatility",
    "nfip_claim_count",
    "nfip_total_claim_payment",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_payment",
    "nfip_avg_claim_payment"
]

for col in log_transform_vars:
    score_df[f"log_{col}"] = np.log1p(score_df[col])

print("Log-transformed variables created:")
for col in log_transform_vars:
    print(f"log_{col}")

Log-transformed variables created:
log_annual_price_volatility
log_price_to_income_ratio
log_median_gross_rent
log_neighbor_avg_price_volatility
log_nfip_claim_count
log_nfip_total_claim_payment
log_nfip_cumulative_claim_payment
log_nfip_recent_3yr_claim_payment
log_nfip_avg_claim_payment


## 9. Define Final Variables for Scoring

Use transformed variables where needed and keep the original variables where transformation is not needed.

In [10]:
# ==================================================
# 9. Define Final Variables for Scoring
# ==================================================

final_score_variables = {
    "housing_pressure": [
        "annual_price_growth_pct",
        "appreciation_from_baseline_pct",
        "log_annual_price_volatility",
        "price_growth_acceleration"
    ],

    "affordability_stress": [
        "log_price_to_income_ratio",
        "log_median_gross_rent",
        "median_household_income"
    ],

    "climate_exposure": [
        "CFLD_RISKS",
        "HRCN_RISKS"
    ],

    "disaster_history": [
        "climate_disaster_count",
        "cumulative_climate_disaster_count",
        "recent_3yr_climate_disaster_count",
        "hurricane_disaster_count"
    ],

    "socioeconomic_vulnerability": [
        "SOVI_SCORE",
        "poverty_rate",
        "unemployment_rate",
        "renter_occupied_share"
    ],

    "resilience_adjustment": [
        "RESL_SCORE"
    ],

    "insurance_loss_stress": [
        "log_nfip_claim_count",
        "log_nfip_total_claim_payment",
        "log_nfip_cumulative_claim_payment",
        "log_nfip_recent_3yr_claim_payment",
        "log_nfip_avg_claim_payment"
    ],

    "spatial_spillover": [
        "neighbor_avg_price_growth_pct",
        "log_neighbor_avg_price_volatility",
        "neighbor_high_growth_share",
        "neighbor_high_volatility_share"
    ]
}

for group, variables in final_score_variables.items():
    print(f"{group}:")
    print(variables)
    print()

housing_pressure:
['annual_price_growth_pct', 'appreciation_from_baseline_pct', 'log_annual_price_volatility', 'price_growth_acceleration']

affordability_stress:
['log_price_to_income_ratio', 'log_median_gross_rent', 'median_household_income']

climate_exposure:
['CFLD_RISKS', 'HRCN_RISKS']

disaster_history:
['climate_disaster_count', 'cumulative_climate_disaster_count', 'recent_3yr_climate_disaster_count', 'hurricane_disaster_count']

socioeconomic_vulnerability:
['SOVI_SCORE', 'poverty_rate', 'unemployment_rate', 'renter_occupied_share']

resilience_adjustment:
['RESL_SCORE']

insurance_loss_stress:
['log_nfip_claim_count', 'log_nfip_total_claim_payment', 'log_nfip_cumulative_claim_payment', 'log_nfip_recent_3yr_claim_payment', 'log_nfip_avg_claim_payment']

spatial_spillover:
['neighbor_avg_price_growth_pct', 'log_neighbor_avg_price_volatility', 'neighbor_high_growth_share', 'neighbor_high_volatility_share']



## 10. Scale Score Variables

Scale all final score variables to a 0–1 range so they can be combined into sub-scores.

In [11]:
# ==================================================
# 10. Scale Score Variables
# ==================================================

from sklearn.preprocessing import MinMaxScaler

scaled_df = score_df.copy()

# Collect final variables used for scoring
final_score_var_list = []

for variables in final_score_variables.values():
    final_score_var_list.extend(variables)

final_score_var_list = list(dict.fromkeys(final_score_var_list))

# Create scaled column names
scaled_score_vars = [f"scaled_{col}" for col in final_score_var_list]

# Apply Min-Max scaling
scaler = MinMaxScaler()

scaled_df[scaled_score_vars] = scaler.fit_transform(
    scaled_df[final_score_var_list]
)

print("Scaled score variables created:")
print(len(scaled_score_vars))

display(
    scaled_df[scaled_score_vars]
    .describe()
    .T
    .round(3)
)

Scaled score variables created:
27


,count,mean,std,min,25%,50%,75%,max
scaled_annual_price_growth_pct,1000.0,0.408,0.160,0.0,0.308,0.400,0.480,1.0
scaled_appreciation_from_baseline_pct,1000.0,0.267,0.215,0.0,0.080,0.207,0.426,1.0
scaled_log_annual_price_volatility,1000.0,0.534,0.149,0.0,0.440,0.538,0.619,1.0
scaled_price_growth_acceleration,1000.0,0.626,0.146,0.0,0.575,0.621,0.698,1.0
scaled_log_price_to_income_ratio,1000.0,0.341,0.156,0.0,0.220,0.336,0.442,1.0
scaled_log_median_gross_rent,1000.0,0.424,0.214,0.0,0.255,0.413,0.557,1.0
scaled_median_household_income,1000.0,0.258,0.160,0.0,0.132,0.224,0.349,1.0
scaled_CFLD_RISKS,1000.0,0.477,0.379,0.0,0.000,0.614,0.789,1.0
scaled_HRCN_RISKS,1000.0,0.767,0.213,0.0,0.632,0.805,0.949,1.0
scaled_climate_disaster_count,1000.0,0.170,0.213,0.0,0.000,0.143,0.286,1.0


## 11. Adjust Protective Variables

Invert protective variables so that higher values always represent higher vulnerability.

In [12]:
# ==================================================
# 11. Adjust Protective Variables
# ==================================================

scored_df = scaled_df.copy()

# Higher income means lower affordability stress
scored_df["scaled_low_income_stress"] = 1 - scored_df["scaled_median_household_income"]

# Higher resilience means lower vulnerability
scored_df["scaled_low_resilience"] = 1 - scored_df["scaled_RESL_SCORE"]

print("Protective variables adjusted:")
print("scaled_low_income_stress")
print("scaled_low_resilience")

display(
    scored_df[
        [
            "scaled_median_household_income",
            "scaled_low_income_stress",
            "scaled_RESL_SCORE",
            "scaled_low_resilience"
        ]
    ]
    .describe()
    .T
    .round(3)
)

Protective variables adjusted:
scaled_low_income_stress
scaled_low_resilience


,count,mean,std,min,25%,50%,75%,max
scaled_median_household_income,1000.0,0.258,0.160,0.0,0.132,0.224,0.349,1.0
scaled_low_income_stress,1000.0,0.742,0.160,0.0,0.651,0.776,0.868,1.0
scaled_RESL_SCORE,1000.0,0.331,0.239,0.0,0.107,0.301,0.493,1.0
scaled_low_resilience,1000.0,0.669,0.239,0.0,0.507,0.699,0.893,1.0


## 12. Create Vulnerability Sub-Scores

Create one score for each vulnerability dimension using the scaled variables.

In [13]:
# ==================================================
# 12. Create Vulnerability Sub-Scores
# ==================================================

# Housing market pressure
scored_df["housing_pressure_score"] = scored_df[
    [
        "scaled_annual_price_growth_pct",
        "scaled_appreciation_from_baseline_pct",
        "scaled_log_annual_price_volatility",
        "scaled_price_growth_acceleration"
    ]
].mean(axis=1)

# Affordability stress
scored_df["affordability_stress_score"] = scored_df[
    [
        "scaled_log_price_to_income_ratio",
        "scaled_log_median_gross_rent",
        "scaled_low_income_stress"
    ]
].mean(axis=1)

# Climate exposure
scored_df["climate_exposure_score"] = scored_df[
    [
        "scaled_CFLD_RISKS",
        "scaled_HRCN_RISKS"
    ]
].mean(axis=1)

# Disaster history
scored_df["disaster_history_score"] = scored_df[
    [
        "scaled_climate_disaster_count",
        "scaled_cumulative_climate_disaster_count",
        "scaled_recent_3yr_climate_disaster_count",
        "scaled_hurricane_disaster_count"
    ]
].mean(axis=1)

# Socioeconomic vulnerability
scored_df["socioeconomic_vulnerability_score"] = scored_df[
    [
        "scaled_SOVI_SCORE",
        "scaled_poverty_rate",
        "scaled_unemployment_rate",
        "scaled_renter_occupied_share"
    ]
].mean(axis=1)

# Resilience adjustment
scored_df["resilience_adjustment_score"] = scored_df["scaled_low_resilience"]

# Insurance-loss stress
scored_df["insurance_loss_stress_score"] = scored_df[
    [
        "scaled_log_nfip_claim_count",
        "scaled_log_nfip_total_claim_payment",
        "scaled_log_nfip_cumulative_claim_payment",
        "scaled_log_nfip_recent_3yr_claim_payment",
        "scaled_log_nfip_avg_claim_payment"
    ]
].mean(axis=1)

# Spatial spillover
scored_df["spatial_spillover_score"] = scored_df[
    [
        "scaled_neighbor_avg_price_growth_pct",
        "scaled_log_neighbor_avg_price_volatility",
        "scaled_neighbor_high_growth_share",
        "scaled_neighbor_high_volatility_share"
    ]
].mean(axis=1)

sub_score_cols = [
    "housing_pressure_score",
    "affordability_stress_score",
    "climate_exposure_score",
    "disaster_history_score",
    "socioeconomic_vulnerability_score",
    "resilience_adjustment_score",
    "insurance_loss_stress_score",
    "spatial_spillover_score"
]

display(
    scored_df[sub_score_cols]
    .describe()
    .T
    .round(3)
)

,count,mean,std,min,25%,50%,75%,max
housing_pressure_score,1000.0,0.459,0.110,0.247,0.389,0.449,0.500,0.857
affordability_stress_score,1000.0,0.502,0.076,0.295,0.448,0.496,0.555,0.787
climate_exposure_score,1000.0,0.622,0.260,0.000,0.390,0.617,0.862,1.000
disaster_history_score,1000.0,0.246,0.227,0.000,0.031,0.219,0.372,1.000
socioeconomic_vulnerability_score,1000.0,0.418,0.127,0.056,0.335,0.419,0.494,0.771
resilience_adjustment_score,1000.0,0.669,0.239,0.000,0.507,0.699,0.893,1.000
insurance_loss_stress_score,1000.0,0.438,0.237,0.000,0.237,0.505,0.619,0.984
spatial_spillover_score,1000.0,0.477,0.252,0.059,0.257,0.456,0.684,0.987


## 13. Create Final Vulnerability Score

Combine the vulnerability sub-scores into one overall climate-housing vulnerability score.

In [14]:
# ==================================================
# 13. Create Final Vulnerability Score
# ==================================================

final_sub_scores = [
    "housing_pressure_score",
    "affordability_stress_score",
    "climate_exposure_score",
    "disaster_history_score",
    "socioeconomic_vulnerability_score",
    "resilience_adjustment_score",
    "insurance_loss_stress_score",
    "spatial_spillover_score"
]

scored_df["climate_housing_vulnerability_score"] = scored_df[
    final_sub_scores
].mean(axis=1)

display(
    scored_df[["climate_housing_vulnerability_score"]]
    .describe()
    .T
    .round(3)
)

,count,mean,std,min,25%,50%,75%,max
climate_housing_vulnerability_score,1000.0,0.479,0.093,0.187,0.409,0.477,0.546,0.782


## 14. Create Vulnerability Classes

Split the final vulnerability score into Low, Medium, and High vulnerability classes.

In [15]:
# ==================================================
# 14. Create Vulnerability Classes
# ==================================================

scored_df["vulnerability_class"] = pd.qcut(
    scored_df["climate_housing_vulnerability_score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

print("Vulnerability class distribution:")
display(scored_df["vulnerability_class"].value_counts().sort_index())

print("\nVulnerability score range by class:")
display(
    scored_df
    .groupby("vulnerability_class")["climate_housing_vulnerability_score"]
    .agg(["count", "min", "mean", "max"])
    .round(3)
)

Vulnerability class distribution:


vulnerability_class
Low       334
Medium    333
High      333
Name: count, dtype: int64


Vulnerability score range by class:


,count,min,mean,max
vulnerability_class,,,,
Low,334,0.187,0.376,0.438
Medium,333,0.439,0.478,0.519
High,333,0.520,0.582,0.782


## 15. Inspect Top High-Vulnerability County-Years

Check the county-years with the highest final vulnerability scores.

In [17]:
# ==================================================
# 15. Inspect Top High-Vulnerability County-Years
# ==================================================

top_vulnerable_county_years = (
    scored_df
    .sort_values("climate_housing_vulnerability_score", ascending=False)
    [
        [
            "RegionName",
            "STCOFIPS",
            "Year",
            "climate_housing_vulnerability_score",
            "vulnerability_class",
            "housing_pressure_score",
            "affordability_stress_score",
            "climate_exposure_score",
            "disaster_history_score",
            "socioeconomic_vulnerability_score",
            "resilience_adjustment_score",
            "insurance_loss_stress_score",
            "spatial_spillover_score"
        ]
    ]
    .head(15)
)

display(top_vulnerable_county_years.round(3))

,RegionName,STCOFIPS,Year,climate_housing_vulnerability_score,vulnerability_class,housing_pressure_score,affordability_stress_score,climate_exposure_score,disaster_history_score,socioeconomic_vulnerability_score,resilience_adjustment_score,insurance_loss_stress_score,spatial_spillover_score
521,Lee County,12071,2022,0.782,High,0.838,0.616,0.988,0.603,0.357,0.915,0.984,0.959
861,Sarasota County,12115,2022,0.749,High,0.815,0.616,0.962,0.633,0.290,0.911,0.800,0.967
641,Miami-Dade County,12086,2022,0.742,High,0.755,0.704,1.000,0.622,0.551,0.558,0.759,0.987
161,Collier County,12021,2022,0.740,High,0.851,0.636,0.974,0.656,0.352,0.575,0.923,0.952
116,Charlotte County,12015,2022,0.738,High,0.827,0.606,0.935,0.591,0.328,0.880,0.796,0.938
191,De Soto County,12027,2022,0.730,High,0.790,0.577,0.595,0.580,0.640,0.992,0.719,0.943
596,Manatee County,12081,2022,0.724,High,0.834,0.624,0.980,0.675,0.304,0.769,0.673,0.934
771,Pinellas County,12103,2022,0.724,High,0.731,0.618,0.977,0.645,0.361,0.893,0.627,0.941
651,Monroe County,12087,2022,0.719,High,0.768,0.787,0.958,0.406,0.374,0.700,0.785,0.975
741,Palm Beach County,12099,2022,0.712,High,0.787,0.639,0.967,0.625,0.370,0.751,0.641,0.914


## 16. Inspect Low-Vulnerability County-Years

Check the county-years with the lowest final vulnerability scores.

In [18]:
# ==================================================
# 16. Inspect Low-Vulnerability County-Years
# ==================================================

low_vulnerable_county_years = (
    scored_df
    .sort_values("climate_housing_vulnerability_score", ascending=True)
    [
        [
            "RegionName",
            "STCOFIPS",
            "Year",
            "climate_housing_vulnerability_score",
            "vulnerability_class",
            "housing_pressure_score",
            "affordability_stress_score",
            "climate_exposure_score",
            "disaster_history_score",
            "socioeconomic_vulnerability_score",
            "resilience_adjustment_score",
            "insurance_loss_stress_score",
            "spatial_spillover_score"
        ]
    ]
    .head(15)
)

display(low_vulnerable_county_years.round(3))

,RegionName,STCOFIPS,Year,climate_housing_vulnerability_score,vulnerability_class,housing_pressure_score,affordability_stress_score,climate_exposure_score,disaster_history_score,socioeconomic_vulnerability_score,resilience_adjustment_score,insurance_loss_stress_score,spatial_spillover_score
15,Baker County,12003,2011,0.187,Low,0.307,0.396,0.000,0.000,0.338,0.256,0.000,0.202
19,Baker County,12003,2015,0.242,Low,0.408,0.411,0.000,0.011,0.316,0.256,0.125,0.410
18,Baker County,12003,2014,0.249,Low,0.391,0.415,0.000,0.031,0.339,0.256,0.249,0.313
17,Baker County,12003,2013,0.256,Low,0.398,0.380,0.000,0.031,0.329,0.256,0.249,0.405
16,Baker County,12003,2012,0.257,Low,0.330,0.393,0.000,0.066,0.329,0.256,0.605,0.081
925,Union County,12125,2011,0.259,Low,0.307,0.329,0.137,0.000,0.454,0.672,0.000,0.169
0,Alachua County,12001,2011,0.261,Low,0.316,0.513,0.436,0.000,0.529,0.160,0.000,0.136
27,Baker County,12003,2023,0.266,Low,0.335,0.465,0.000,0.418,0.179,0.256,0.144,0.333
955,Wakulla County,12129,2011,0.269,Low,0.290,0.407,0.504,0.000,0.204,0.565,0.000,0.182
480,Lafayette County,12067,2011,0.271,Low,0.299,0.295,0.305,0.000,0.310,0.884,0.000,0.074


## 17. Save Vulnerability-Ready Dataset

Save the dataset with vulnerability sub-scores, final vulnerability score, and vulnerability class.

In [19]:
# ==================================================
# 17. Save Vulnerability-Ready Dataset
# ==================================================

OUTPUT_DATA_PATH = (
    INTERIM_DATA
    / "county_year_vulnerability_scores.csv"
)

scored_df.to_csv(OUTPUT_DATA_PATH, index=False)

print("Vulnerability-ready dataset saved.")
print(f"Saved to: {OUTPUT_DATA_PATH}")
print(f"Shape: {scored_df.shape}")

Vulnerability-ready dataset saved.
Saved to: ..\data\interim\county_year_features\county_year_vulnerability_scores.csv
Shape: (1000, 123)
